# Exercise 3 — Aggregations: exact vs approx COUNT(DISTINCT) (200M events)

**Workload:** `groupBy(title_id, country_code)` with `count_distinct(user_id)` vs
`approx_count_distinct(user_id)` (HLL, rsd=0.05) on 200M events, ~1M distinct users.

## Exact plan: dedup via key expansion
Four HashAggregates around one Exchange:
1. Partial dedup with **user_id IN the grouping key** `[title_id, country_code, user_id]`;
2. Exchange: **10.6 MiB / 231K rows** (200M → 231K before the shuffle, ~1000×
   reduction by partial aggregation);
3. Merge dedup across partitions; 4. key collapses back, distinct becomes a plain count.

## Approx (HLL) plan: dedup via sketches
- **user_id disappears from the grouping keys** — it goes inside the aggregate:
  `partial_approx_count_distinct(user_id, 0.05) AS buffer` — a binary HLL sketch
  per group;
- **Shorter pipeline:** one `finalmerge_approx_count_distinct(merge buffer)` after
  the shuffle instead of the exact plan's three-HashAggregate cascade. Sketches are
  mergeable (bucket-wise max of HLL registers), so merging is one pass.

## The elephant in the room: approx shuffled MORE
| | rows shuffled | bytes shuffled |
|---|---|---|
| exact | 231K | **10.6 MiB** |
| approx (HLL) | 176K | **80.6 MiB (8× more!)** |

**Why:** one user_id (Long) is 8 bytes; one HLL sketch at rsd=0.05 is ~KBs —
*fixed size regardless of how many values it holds*. With ~1M users over 200M
events, many (title, country) groups had 1–2 viewers; exact shipped 8-byte values,
HLL shipped multi-KB sketches per group. Low per-group cardinality → sketch
overhead dominates.

## Verdict
**On this data, exact won** — partial aggregation crushed the volume and the
shuffle stayed tiny. **HLL's strength is its ceiling:** if distinct users grew from
1M to 500M, the exact shuffle would balloon to tens of GiB (OOM risk), while the
HLL shuffle would stay ~80 MiB — sketch size does not depend on cardinality.

**Rules of thumb:**
- exact: low-to-mid cardinality relative to volume; audit/finance (exactness required);
- approx: high per-group cardinality, dashboards, and incremental/mergeable distinct
  (Spark 3.5+ `hll_sketch_agg` persists sketches for reaggregation);
- rsd < 0.01 → use exact (sketch grows past the win).

Sketch family: HLL = "how many distinct"; Count-Min = "how often is this key"
(probabilistic heavy-hitter detection — an alternative to our salting detection
pass); Bloom = "have I seen this".

In [0]:
from conf.config import SILVER_EVENTS
from pyspark.sql import functions as F


events = spark.table(SILVER_EVENTS)   # 200M rows, ~1M distinct user_id

# ---- Variant A: EXACT distinct ----
exact = (events.groupBy("title_id", "country_code")
    .agg(F.count_distinct("user_id").alias("distinct_users"),
         F.count("*").alias("events")))
exact.write.format("noop").mode("overwrite").save()

# ---- Variant B: APPROX distinct (default 5% rsd) ----
approx5 = (events.groupBy("title_id", "country_code")
    .agg(F.approx_count_distinct("user_id").alias("distinct_users"),
         F.count("*").alias("events")))
approx5.write.format("noop").mode("overwrite").save()

# ---- Variant C: NO distinct (additive-only floor) ----
nodist = (events.groupBy("title_id", "country_code")
    .agg(F.count("*").alias("events")))
nodist.write.format("noop").mode("overwrite").save()

In [0]:
e = (events.groupBy("title_id", "country_code")
     .agg(F.count_distinct("user_id").alias("d")).agg(F.sum("d")).collect()[0][0])
a = (events.groupBy("title_id", "country_code")
     .agg(F.approx_count_distinct("user_id").alias("d")).agg(F.sum("d")).collect()[0][0])
print(f"exact={e:,}  approx={a:,}  error={abs(a-e)/e*100:.2f}%")